In [ ]:
from openai import OpenAI
import json
import xml.etree.ElementTree as ET
import pandas as pd
import re
import random
import tiktoken

In [ ]:
client = OpenAI(
  api_key='ADD-YOUR-API-KEY'
)

In [ ]:
file_path = '../data/IQVIA/filtered_trial_outcomes_with_labels.csv'
df = pd.read_csv(file_path)

In [ ]:
df

In [ ]:
def element_to_dict(el):
    
    children = list(el)
    if not children:
        return el.text
    result = {}
    for child in children:
        child_dict = element_to_dict(child)
        if child.tag in result:
            if not isinstance(result[child.tag], list):
                result[child.tag] = [result[child.tag]]
            result[child.tag].append(child_dict)
        else:
            result[child.tag] = child_dict
            
    return result

def xml_to_dict(element):
    
    return {element.tag: element_to_dict(element)}

def read_xml_file(file_path):
    
    tree = ET.parse(file_path)
    root = tree.getroot()
    return xml_to_dict(root)

In [ ]:
def extract_intervention_name(study_id):
    
    xml_path = f'../data/trials/{study_id[:7]}xxxx/{study_id}.xml'
    xml_dict = read_xml_file(xml_path)
    xml_string = json.dumps(xml_dict)
    pattern = r'"intervention_name":\s*"([^"]*)"'
    match = re.search(pattern, xml_string)
    
    if match:
        intervention_name = match.group(1).lower()
        return intervention_name
        

In [ ]:
def num_tokens_from_string(str):
    encoding = tiktoken.encoding_for_model("gpt-4o-mini")
    num_tokens = len(encoding.encode(str))
    return num_tokens

In [ ]:
df['intervention_name'] = df['studyid'].apply(extract_intervention_name)

In [ ]:
df

In [ ]:
drug_name_file_path = '../data/raw/drugbank vocabulary.csv'
drug_name_df = pd.read_csv(drug_name_file_path)

drug_name_list = drug_name_df['Common name'].tolist()
drug_name_list_lowercased = [name.lower() for name in drug_name_list]
drug_set = set(drug_name_list_lowercased )

In [ ]:
df_filtered = df[df['intervention_name'].isin(drug_set)]

In [ ]:
df_filtered

In [ ]:
label_counts = df_filtered.groupby(['intervention_name', 'label']).size().unstack(fill_value=0)

# Filter for intervention names where both label 0 and label 1 have at least 3 occurrences
valid_interventions = label_counts[(label_counts[0] >= 3) & (label_counts[1] >= 3)].index

# Filter the original DataFrame based on the valid intervention names
df_filtered_final = df_filtered[df_filtered['intervention_name'].isin(valid_interventions)]

In [ ]:
df_filtered_final

In [ ]:
# Group by intervention_name and label, then count occurrences in df_filtered_final
label_counts_final = df_filtered_final.groupby(['intervention_name', 'label']).size().unstack(fill_value=0)

# Display the counts for each intervention name
print(label_counts_final)

In [ ]:
intervention_names = label_counts_final.index.tolist()
print(intervention_names)

In [ ]:
def draw_random_intervention():

    random_intervention = random.choice(intervention_names)
    random_label = random.randint(0, 1)

    return random_intervention, random_label

In [ ]:
def get_reports_for_intervention(intervention_name, label):

    eligible_rows = df[(df['intervention_name'] == intervention_name) & (df['label'] == label)]
    selected_rows = eligible_rows.sample(n=3, random_state=42)
    xml_strings = []
    
    for _, row in selected_rows.iterrows():
        study_id = row['studyid']
        xml_path = f'../data/trials/{study_id[:7]}xxxx/{study_id}.xml'
        xml_dict = read_xml_file(xml_path)
        xml_string = json.dumps(xml_dict)
        xml_strings.append(xml_string)
        
    return xml_strings

In [ ]:
def get_reports():
    
    while True:
        random_intervention, random_label = draw_random_intervention()
        string_list = get_reports_for_intervention(random_intervention, random_label)
        
        total_tokens = 0
        for xml_string in string_list:
            total_tokens += num_tokens_from_string(xml_string)

        if total_tokens <= 100_000:
            return string_list, random_intervention, random_label

In [ ]:
reason_context_prompt = "You are now a medical expert in the clinical area. You are given information of a medical intervention, and three clinical trial reports of it, either all successful or all failed. You are asked to analyze these input and write reasons resulting the trials' success/failure. Your writing style must be consistent within the clinical study. You must ensure that your language is precise, technical, and formal."
reason_data_generation_prompt = "Write 5 reasons leading {name} to {type} in these trials. Remember to make sure that your language is precise, technical, and formal. Be creative and write unique reasons."
reason_format_prompt = "Your output should strictly follow the following format: \n 1. (...) \n 2. (...) \n 3. (...) \n 4. (...) \n 5. (...), with (...) being the reasons you write."
reason_diversity_prompt = "Can you provide something more diverse compared to the previously generated reasons?"
reason_types = ["Failed", "Successful"]
reason_types_verb = ["fail", "succeed"]

def generate_reasons(examples, intervention_name, label):
    
    reason_messages = [
        {"role": "system", "content": reason_context_prompt}
    ]
    reason_user_messages = []
    
    for text in examples:
        reason_example_prompt = f"{types[label]} clinical trial of {intervention_name}: {text}"
        reason_user_messages.append({"role": "user", "content": reason_example_prompt})
        
    reason_user_messages.append({"role": "user", "content": reason_data_generation_prompt.format(name=intervention_name, type=reason_types_verb[label])})
    reason_user_messages.append({"role": "user", "content": reason_format_prompt})
    reason_user_messages.append({"role": "user", "content": reason_diversity_prompt})

    reason_concatenated_messages = " \n".join([message["content"] for message in reason_user_messages])
    reason_messages.append({"role": "user", "content": reason_concatenated_messages})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=reason_messages,
        temperature=1
    )
    reason_output_message = response.choices[0].message.content

    
    return reason_output_message

In [ ]:
context_prompt = "You are now a medical expert in the clinical area. You are provided with reasons of a medical intervention that might lead to the success/failure of a clinical trial, with three real clinical trials that are either all successful/failed. You are asked to write a clinical trial for the intervention of the same label (failure/success). Your writing style must be consistent with the real clinical trial examples. You must ensure that your language is precise, technical, and formal."
reason_prompt = "Here are five reasons that could lead to the {type} of clinical trials of {name}: {reasons} "
data_generation_prompt = "Write a report of a {type} clinical trial of {name}. Remember to make sure that your language is precise, technical, and formal. Be creative and write unique reports."
constraint_prompt = "Your style of output should be strictly similar to the xml-like format of the provided three clinical trial examples, but you cannot simply modify or rewrite them. The name of the intervention must be {name}, and you must refer to the reasons when writing clinical trials."
diversity_prompt = "Can you provide something more diverse compared to the previously generated trials?"
types_noun = ["failure", "success"]
types = ["Failed", "Successful"]

def generate_trial():
    
    messages = [
        {"role": "system", "content": context_prompt}
    ]
    user_messages = []

    examples, intervention_name, label = get_reports()
    
    for text in examples:
        example_prompt = f"{types[label]} clinical trial of {intervention_name}  \n: {text}"
        user_messages.append({"role": "user", "content": example_prompt})
    
    user_messages.append({"role": "user", "content": constraint_prompt.format(name=intervention_name)})
    user_messages.append({"role": "user", "content": reason_prompt.format(type=types_noun[label], name=intervention_name, reasons=generate_reasons(examples, intervention_name, label))})
    user_messages.append({"role": "user", "content": data_generation_prompt.format(type=types[label], name=intervention_name)})
    user_messages.append({"role": "user", "content": diversity_prompt})

    concatenated_messages = " \n".join([message["content"] for message in user_messages])
    #return concatenated_messages
    
    messages.append({"role": "user", "content": concatenated_messages})
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=1
    )

    output_message = response.choices[0].message.content
    
    return output_message, intervention_name, label

In [ ]:
label_list = []
count = 0
correct_intervention_list = []
generated_intervention_list = []
correct_count = 0

while count < 5000:

    output_message, random_intervention, random_label = generate_trial()
    correct_intervention_list.append(random_intervention)
    pattern = r'"intervention_name":\s*"([^"]*)"'
    match = re.search(pattern, output_message)
    if match:
        intervention_name = match.group(1).lower()
        generated_intervention_list.append(intervention_name)
        if intervention_name == random_intervention:
            correct_count += 1
    else:
        generated_intervention_list.append(0)
            
    file_name = f"../data/synthetic/retrieval_reasoning_reports/synthetic_clinical_report_{count}.txt"
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(output_message)

    label_list.append(random_label)
    count += 1

In [ ]:
with open("../data/synthetic/retrieval_reasoning_label.txt", "a") as label_file:
    for label in label_list:
        label_file.write(f"{label}\n")

In [ ]:
with open("../data/synthetic/correct_intervention_list.txt", "a") as name_file:
    for intervention in correct_intervention_list:
        name_file.write(f"{intervention}\n")

In [ ]:
with open("../data/synthetic/generated_intervention_list.txt", "a") as name_file:
    for intervention in generated_intervention_list:
        name_file.write(f"{intervention}\n")